# Práctica 3.1 Entrenar y servir un bot
Gustavo Adolfo Martínez Enríquez

Se usa CartPole como ejemplo de un bot. Se comparan CPU y GPU con el mismo modelo y semilla. Para medir CUDA, elegir **Entorno de ejecución > Cambiar tipo de entorno de ejecución > T4 GPU** en Colab.

In [1]:
%pip -q install gymnasium==1.4.0 stable-baselines3==2.9.2

In [2]:
"""Práctica 3.1: entrenamiento PPO y medición real de inferencia en CPU."""
import argparse
import json
import math
import platform
import time
from pathlib import Path

import gymnasium as gym
import numpy as np
import torch
from stable_baselines3 import PPO
from stable_baselines3.common.evaluation import evaluate_policy


def benchmark(device, steps, output):
    torch.set_num_threads(1)
    env = gym.make("CartPole-v1")
    env.reset(seed=42)
    model = PPO("MlpPolicy", env, seed=42, device=device, verbose=0)
    start = time.perf_counter()
    model.learn(total_timesteps=steps)
    elapsed = time.perf_counter() - start
    actual_steps = model.num_timesteps
    output.mkdir(parents=True, exist_ok=True)
    model.save(output / f"bot_nebula_{device}")
    # Inferencia en CPU incluso cuando el entrenamiento se hizo en CUDA.
    model.policy.to("cpu")
    model.device = torch.device("cpu")
    observations = []
    obs, _ = env.reset(seed=99)
    for _ in range(100):
        observations.append(obs.copy())
        obs, _, terminated, truncated, _ = env.step(env.action_space.sample())
        if terminated or truncated:
            obs, _ = env.reset()
    for obs in observations:
        model.predict(obs, deterministic=True)
    durations = []
    for i in range(10000):
        start = time.perf_counter_ns()
        model.predict(observations[i % len(observations)], deterministic=True)
        durations.append((time.perf_counter_ns() - start) / 1e6)
    mean_ms = float(np.mean(durations))
    capacity = 1000 / mean_ms
    reward, std = evaluate_policy(model, env, n_eval_episodes=10, deterministic=True, warn=False)
    result = {
        "dispositivo_entrenamiento": device, "equipo": platform.platform(),
        "torch": torch.__version__, "pasos_solicitados": steps, "pasos_reales": actual_steps,
        "semilla": 42, "hilos_cpu": 1, "entrenamiento_s": elapsed,
        "inferencia_cpu_media_ms": mean_ms, "inferencia_cpu_p95_ms": float(np.percentile(durations, 95)),
        "decisiones_s_por_hilo": capacity, "demanda_bots_s": 2000 * 10,
        "vCPU_con_margen_30_por_ciento": math.ceil(20000 / capacity * 1.3),
        "recompensa_media_10_episodios": float(reward), "recompensa_desviacion": float(std),
        "fecha_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        "nota": "Una medición local de un hilo no equivale a medir una vCPU de Google Cloud."
    }
    (output / f"resultado_{device}.json").write_text(json.dumps(result, indent=2, ensure_ascii=False))
    print(json.dumps(result, indent=2, ensure_ascii=False))
    env.close()




Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [3]:
from pathlib import Path
benchmark("cpu", 100000, Path("resultados"))
if torch.cuda.is_available():
    benchmark("cuda", 100000, Path("resultados"))
else:
    print("GPU CUDA no disponible. Falta ejecutar la comparación en un entorno con T4.")

{
  "dispositivo_entrenamiento": "cpu",
  "equipo": "Linux-6.6.159+-x86_64-with-glibc2.39",
  "torch": "2.14.1+cu130",
  "pasos_solicitados": 100000,
  "pasos_reales": 100352,
  "semilla": 42,
  "hilos_cpu": 1,
  "entrenamiento_s": 120.84842659700007,
  "inferencia_cpu_media_ms": 0.3520138853,
  "inferencia_cpu_p95_ms": 0.51913665,
  "decisiones_s_por_hilo": 2840.7970303437346,
  "demanda_bots_s": 20000,
  "vCPU_con_margen_30_por_ciento": 10,
  "recompensa_media_10_episodios": 500.0,
  "recompensa_desviacion": 0.0,
  "fecha_utc": "2026-11-10T02:31:23Z",
  "nota": "Una medición local de un hilo no equivale a medir una vCPU de Google Cloud."
}


/usr/local/lib/python3.13/dist-packages/stable_baselines3/common/on_policy_algorithm.py:150: UserWarning: You are trying to run PPO on the GPU, but it is primarily intended to run on the CPU when not using a CNN policy (you are using ActorCriticPolicy which should be a MlpPolicy). See https://github.com/DLR-RM/stable-baselines3/issues/1245 for more info. You can pass `device='cpu'` or `export CUDA_VISIBLE_DEVICES=` to force using the CPU.Note: The model will train, but the GPU utilization will be poor and the training might take longer than on CPU.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


{
  "dispositivo_entrenamiento": "cuda",
  "equipo": "Linux-6.6.159+-x86_64-with-glibc2.39",
  "torch": "2.14.1+cu130",
  "pasos_solicitados": 100000,
  "pasos_reales": 100352,
  "semilla": 42,
  "hilos_cpu": 1,
  "entrenamiento_s": 217.24824020300002,
  "inferencia_cpu_media_ms": 0.3463975472,
  "inferencia_cpu_p95_ms": 0.48207364999999996,
  "decisiones_s_por_hilo": 2886.8564690575845,
  "demanda_bots_s": 20000,
  "vCPU_con_margen_30_por_ciento": 10,
  "recompensa_media_10_episodios": 500.0,
  "recompensa_desviacion": 0.0,
  "fecha_utc": "2026-11-10T02:35:07Z",
  "nota": "Una medición local de un hilo no equivale a medir una vCPU de Google Cloud."
}


In [4]:
import pandas as pd
from IPython.display import display
rows = [json.loads(p.read_text()) for p in Path("resultados").glob("resultado_*.json")]
display(pd.DataFrame(rows)[["dispositivo_entrenamiento", "entrenamiento_s", "inferencia_cpu_media_ms", "decisiones_s_por_hilo", "vCPU_con_margen_30_por_ciento"]])

,dispositivo_entrenamiento,entrenamiento_s,inferencia_cpu_media_ms,decisiones_s_por_hilo,vCPU_con_margen_30_por_ciento
0,cuda,217.248240,0.346398,2886.856469,10
1,cpu,120.848427,0.352014,2840.797030,10


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


## Dimensionamiento
La demanda es 2,000 bots por 10 decisiones por segundo: **20,000 decisiones/s**. Se divide entre la capacidad medida y se agrega 30% de margen. La estimación local se debe confirmar en la VM elegida.

Con una red pequeña, la GPU puede tardar más por el costo de transferir datos. La comparación final depende de las dos mediciones reales.

El archivo JSON guarda los tiempos para elaborar la tabla del reporte. El precio de la VM se consulta en la calculadora oficial de Google Cloud: https://cloud.google.com/products/calculator